# EB-JEPA — Colab GPU First Run

This notebook is designed for the official Colab extension in VS Code. Run the cells in order. It starts with a Video JEPA smoke test, then runs a small training job and creates visual rollouts. An optional AC-JEPA experiment is included at the end.

## 1. Check the GPU and runtime

In [2]:
import platform
import subprocess

print('Python:', platform.python_version())
subprocess.run(['nvidia-smi'], check=True)

Python: 3.13.15


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [3]:
import torch

assert torch.cuda.is_available(), 'No GPU found. Reconnect with Select Kernel > Colab > GPU.'
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4


## 2. Mount Google Drive and define working paths

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from pathlib import Path

GITHUB_USER = 'Furkan-Coban'
REPO_URL = f'https://github.com/{GITHUB_USER}/eb_jepa.git'
REPO_DIR = Path('/content/eb_jepa')
RUN_ROOT = Path('/content/eb_jepa_runs')
DRIVE_ROOT = Path('/content/drive/MyDrive/eb_jepa_artifacts')
DRIVE_DATA = Path('/content/drive/MyDrive/eb_jepa_data')

RUN_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_DATA.mkdir(parents=True, exist_ok=True)
print('Repo:', REPO_URL)
print('Artifacts:', DRIVE_ROOT)

Repo: https://github.com/Furkan-Coban/eb_jepa.git
Artifacts: /content/drive/MyDrive/eb_jepa_artifacts


## 3. Clone the fork and install the environment

In [6]:
import os
import subprocess

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

Commit: 6e9d9ae


In [7]:
!pip -q install uv
!uv python install 3.12
!uv sync

Python 3.12 is already installed
Resolved 126 packages in 1ms
Uninstalled 1 package in 1ms
Installed 1 package in 1ms                                  
 ~ decord==0.6.0


In [8]:
!uv run python -c "import torch; print('uv env CUDA:', torch.cuda.is_available()); print('uv env GPU:', torch.cuda.get_device_name(0))"

Uninstalled 1 package in 1ms
Installed 1 package in 2ms                                  
uv env CUDA: True
uv env GPU: Tesla T4


## 4. Restore the Moving MNIST cache from Drive if available

In [9]:
import shutil

dataset_name = 'mnist_test_seq.npy'
drive_dataset = DRIVE_DATA / dataset_name
repo_dataset = REPO_DIR / 'datasets' / dataset_name
repo_dataset.parent.mkdir(parents=True, exist_ok=True)
if drive_dataset.exists() and not repo_dataset.exists():
    shutil.copy2(drive_dataset, repo_dataset)
    print('Dataset restored from the Drive cache.')
else:
    print('No cache found or the dataset is already present; the first run will download it if needed.')

No cache found or the dataset is already present; the first run will download it if needed.


## 5. Video JEPA smoke test

This run only verifies the data, forward/backward, and checkpoint pipeline. Do not interpret it as a learning result.

In [10]:
SMOKE_DIR = RUN_ROOT / "video_smoke"

if SMOKE_DIR.exists():
    shutil.rmtree(SMOKE_DIR)

!uv run python -m examples.video_jepa.main \
--fname=examples/video_jepa/cfgs/default.yaml \
--folder={SMOKE_DIR} \
--logging.log_wandb=False \
--logging.log_every=1 \
--data.train_size=512 \
--data.val_size=128 \
--data.batch_size=32 \
--data.num_workers=2 \
--model.steps=1 \
--optim.epochs=1

Uninstalled 1 package in 1ms
Installed 1 package in 2ms                                  
[INFO    ][2026-10-02 13:37:00][eb_jepa.training_utils][load_config              ] Loaded config from examples/video_jepa/cfgs/default.yaml
[INFO    ][2026-10-02 13:37:00][eb_jepa.training_utils][load_config              ] Applied 8 config overrides
[INFO    ][2026-10-02 13:37:00][eb_jepa.training_utils][setup_device             ] Using device: cuda
[INFO    ][2026-10-02 13:37:00][eb_jepa.training_utils][setup_seed               ] Random seed set to 2025
[INFO    ][2026-10-02 13:37:00][eb_jepa.training_utils][setup_wandb              ] W&B logging disabled
[INFO    ][2026-10-02 13:37:02][eb_jepa.training_utils][log_data_info            ] 📦 Data: MovingMNIST | 16 batches x 32 samples | train=512 | val=128
[INFO    ][2026-10-02 13:37:02][__main__            ][run                      ] Initializing model...
[INFO    ][2026-10-02 13:37:03][eb_jepa.training_utils][log_model_info           ] 🧠 Model:
J

In [13]:
assert (SMOKE_DIR / 'latest.pth.tar').exists(), 'The smoke checkpoint was not created.'
print('Smoke test completed:', SMOKE_DIR / 'latest.pth.tar')

Smoke test completed: /content/eb_jepa_runs/video_smoke/latest.pth.tar


## 6. Video JEPA mini learning run

The safe T4 profile uses `batch_size=32` and one training prediction step. Multi-step rollout is still evaluated after training. If this profile runs out of memory, reduce the batch size to 16.

In [ ]:
MINI_DIR = RUN_ROOT / "video_mini"

if MINI_DIR.exists():
    shutil.rmtree(MINI_DIR)

!uv run python -m examples.video_jepa.main \
--fname=examples/video_jepa/cfgs/default.yaml \
--folder={MINI_DIR} \
--logging.log_wandb=False \
--logging.log_every=1 \
--logging.save_every=1 \
--data.train_size=4000 \
--data.val_size=512 \
--data.batch_size=32 \
--data.num_workers=2 \
--model.steps=1 \
--optim.epochs=5

Uninstalled 1 package in 1ms
Installed 1 package in 3ms                                  
[INFO    ][2026-10-02 13:37:14][eb_jepa.training_utils][load_config              ] Loaded config from examples/video_jepa/cfgs/default.yaml
[INFO    ][2026-10-02 13:37:14][eb_jepa.training_utils][load_config              ] Applied 9 config overrides
[INFO    ][2026-10-02 13:37:14][eb_jepa.training_utils][setup_device             ] Using device: cuda
[INFO    ][2026-10-02 13:37:14][eb_jepa.training_utils][setup_seed               ] Random seed set to 2025
[INFO    ][2026-10-02 13:37:14][eb_jepa.training_utils][setup_wandb              ] W&B logging disabled
[INFO    ][2026-10-02 13:37:18][eb_jepa.training_utils][log_data_info            ] 📦 Data: MovingMNIST | 125 batches x 32 samples | train=4,000 | val=512
[INFO    ][2026-10-02 13:37:18][__main__            ][run                      ] Initializing model...
[INFO    ][2026-10-02 13:37:19][eb_jepa.training_utils][log_model_info           ] 🧠 Model

## 7. Loss grafikleri

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

with open(MINI_DIR / 'metrics.jsonl') as f:
    metrics = pd.DataFrame(json.loads(line) for line in f if line.strip())
display(metrics)

plot_columns = [c for c in ['train/pred_loss', 'train/vc_loss', 'val/recon_loss', 'val/det_loss'] if c in metrics]
ax = metrics.plot(x='epoch', y=plot_columns, marker='o', figsize=(10, 5), grid=True)
ax.set_title('Video JEPA mini-run metrics')
ax.set_ylabel('Loss')
figure_path = MINI_DIR / 'loss_curves.png'
plt.tight_layout()
plt.savefig(figure_path, dpi=150)
plt.show()
print('Saved:', figure_path)

## 8. Ground truth / predicted rollout / detection GIF'i

The GIF rows show ground truth, the decoded predicted rollout, and the digit-detection overlay, respectively.

In [ ]:
VIZ_DIR = MINI_DIR / 'visualizations'
!uv run python -m examples.video_jepa.visualize \
  --checkpoint {MINI_DIR / 'latest.pth.tar'} \
  --output_dir {VIZ_DIR} \
  --num_samples 2

In [ ]:
from IPython.display import Image, display

for gif_path in sorted(VIZ_DIR.glob('*.gif')):
    print(gif_path.name)
    display(Image(filename=str(gif_path)))

print((VIZ_DIR / 'tensor_shapes.json').read_text())

## 9. Back up the dataset and results to Drive

In [ ]:
if repo_dataset.exists() and not drive_dataset.exists():
    shutil.copy2(repo_dataset, drive_dataset)
    print('Moving MNIST copied to the Drive cache.')

drive_run = DRIVE_ROOT / 'video_mini'
if drive_run.exists():
    shutil.rmtree(drive_run)
shutil.copytree(MINI_DIR, drive_run)
print('Results saved:', drive_run)

## 10. Optional second experiment: Action-Conditioned JEPA

First confirm that the Video JEPA outputs above are correct. This cell disables planning and produces a small action-conditioned checkpoint.

In [ ]:
AC_DIR = RUN_ROOT / 'ac_video_mini'
if AC_DIR.exists():
    shutil.rmtree(AC_DIR)
!MPLBACKEND=Agg uv run python -m examples.ac_video_jepa.main \
  --fname=examples/ac_video_jepa/cfgs/train.yaml \
  --folder={AC_DIR} \
  --logging.log_wandb=False \
  --meta.load_model=False \
  --meta.enable_plan_eval=False \
  --model.compile=False \
  --training.dtype=float16 \
  --data.size=2048 \
  --data.val_size=128 \
  --data.batch_size=32 \
  --data.num_workers=2 \
  --model.nsteps=2 \
  --optim.epochs=3

### Correct-action vs shuffled-action check

The green trajectory uses the correct actions, the red trajectory uses batch-shuffled actions, and the black trajectory is ground truth.

In [ ]:
AC_VIZ_DIR = AC_DIR / 'action_sanity'
!MPLBACKEND=Agg uv run python -m examples.ac_video_jepa.action_sanity \
  --checkpoint={AC_DIR / 'latest.pth.tar'} \
  --output_dir={AC_VIZ_DIR} \
  --nsteps=8 \
  --num_samples=128 \
  --num_plot_samples=4

In [ ]:
display(Image(filename=str(AC_VIZ_DIR / 'action_trajectory_comparison.png')))
print((AC_VIZ_DIR / 'action_sanity_metrics.json').read_text())

In [ ]:
drive_ac = DRIVE_ROOT / 'ac_video_mini'
if drive_ac.exists():
    shutil.rmtree(drive_ac)
shutil.copytree(AC_DIR, drive_ac)
print('AC-JEPA checkpoint saved:', drive_ac)